In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

np.set_printoptions(suppress=True, formatter={'float': '{:0.4f}'.format})

KCAL_TO_KJ     = 4.184
PMF_TOTAL      = "../data/pmf_data/total"
DIST_TOTAL_DIR = "../data/distribution_data/total"    # summary_{analog}.dat
US_DIR         = "../data/SUPP_umbrella_sampling/pmf"

ANALOGS = ['scc', 'scs', 'scy', 'scd']
LABELS  = {'scc': 'CYS', 'scs': 'SER',
           'scy': 'TYR', 'scd': 'ASP$^-$'}

REGION_BOUNDS = [9.5, 19.5, 30.0]

PMF_COLOR = 'tab:blue'
US_COLOR  = '#8e44ad'    # violet

# ─── Loaders ────────────────────────────────────────────────
def load_our_pmf(analog):
    path = os.path.join(PMF_TOTAL, analog, f"pmf_{analog}.dat")
    if not os.path.isfile(path):
        return None, None, None
    df = pd.read_csv(path, sep=r'\s+')
    return df['z'].to_numpy(), df['mean'].to_numpy(), df['se'].to_numpy()

def load_us(analog):
    path = os.path.join(US_DIR, f"{analog}/pmf-us-{analog}.dat")
    if not os.path.isfile(path):
        return None, None, None
    arr = np.loadtxt(path, comments='#', dtype=float)
    z, pmf = arr[:, 0], arr[:, 1] * KCAL_TO_KJ
    se = arr[:, 2] * KCAL_TO_KJ if arr.shape[1] >= 3 else None
    return z, pmf, se

def plateau_end(analog, z_max=20.0):
    """Return the upper z bound of the null-density plateau starting at z=0,
    looking up to z_max. Returns 0 if there is no plateau."""
    path = os.path.join(DIST_TOTAL_DIR, analog, f"summary_{analog}.dat")
    if not os.path.isfile(path):
        return 0.0
    df = pd.read_csv(path, sep=r'\s+')
    df = df[(df['z'] >= 0) & (df['z'] <= z_max)].sort_values('z').reset_index(drop=True)
    end = 0.0
    for _, row in df.iterrows():
        if row['mean'] == 0:
            end = row['z'] + 0.5
        else:
            break
    return end

def plot_pmf_with_plateau(ax, x, y, pe, color, lw, label=None, alpha=1.0):
    """Plot y vs x with a dashed segment on the null-density plateau (x ≤ pe)
    and a solid segment beyond it (x ≥ pe), sharing the same color. The
    legend label is attached only to the solid segment."""
    x = np.asarray(x)
    y = np.asarray(y)
    if pe > 0:
        mask_dash  = x <= pe
        mask_solid = x >= pe   # overlap at pe keeps the curve continuous
    else:
        mask_dash  = np.zeros_like(x, dtype=bool)
        mask_solid = np.ones_like(x, dtype=bool)
    if mask_solid.any():
        ax.plot(x[mask_solid], y[mask_solid], '-',
                color=color, lw=lw, alpha=alpha, label=label)
    if mask_dash.any():
        ax.plot(x[mask_dash], y[mask_dash], '--',
                color=color, lw=lw, alpha=alpha)

# ─── Plot ───────────────────────────────────────────────────
#n_cols, n_rows = 2, 3
n_cols, n_rows = 2, 2
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(12, 2.2 * n_rows),
                          sharex=True, dpi=2000,
                          gridspec_kw={'hspace': 0.35, 'wspace': 0.20})
axes_flat = axes.flatten()

first_populated = None
for idx, analog in enumerate(ANALOGS):
    ax = axes_flat[idx]
    plotted = False

    z, y, e = load_our_pmf(analog)
    if z is not None:
        pe = plateau_end(analog)
        plot_pmf_with_plateau(ax, z, y, pe,
                              color=PMF_COLOR, lw=1.6, label='Boltzmann T.')
        if e is not None:
            ax.fill_between(z, y - e, y + e, color=PMF_COLOR,
                            alpha=0.25, linewidth=0)
        plotted = True

    zu, yu, eu = load_us(analog)
    if zu is not None:
        ax.plot(zu, yu, color=US_COLOR, lw=1.6, linestyle='--',
                label='Umbrella Sampling')
        if eu is not None:
            ax.fill_between(zu, yu - eu, yu + eu, color=US_COLOR,
                            alpha=0.20, linewidth=0)
        plotted = True
    else:
        ax.text(0.98, 0.05, 'no US data',
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=8, color='gray')

    for xv in REGION_BOUNDS:
        ax.axvline(x=xv, linestyle='--', alpha=0.7, linewidth=0.8,
                   color='gray', zorder=0)

    ax.set_xlim(0, 35)
    ax.axhline(0.0, color='k', lw=0.5, alpha=0.5)
    ax.set_title(LABELS[analog], fontsize=10, fontweight='bold')
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.grid(True, which='major', linestyle='--', alpha=0.4, linewidth=0.3)
    ax.tick_params(axis='both', labelsize=8)

    if plotted and first_populated is None:
        first_populated = ax

if first_populated is not None:
    h, l = first_populated.get_legend_handles_labels()
    first_populated.legend(h, l, fontsize=8, frameon=False, loc='best')

fig.text(0.5, 0.02, r"z (Å)", ha='center', fontsize=11)
fig.text(0.06, 0.5, "PMF (kJ/mol)", va='center',
         rotation='vertical', fontsize=11)

os.makedirs("../plot", exist_ok=True)
out = "../plot/FigureS10.png"
plt.tight_layout(rect=[0.04, 0.03, 1, 1])
plt.savefig(out, dpi=600, bbox_inches='tight')
plt.show()
print("Saved →", out)
